In [ ]:
# uncomment these and run this cell if needed
!pip install evaluate
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset, Dataset
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import get_scheduler
from transformers import Trainer, TrainingArguments
from datasets import concatenate_datasets
import evaluate
import numpy as np
import requests

In [ ]:
#initalize variables
header1 = "Passage: "
header2 = "Question: "
header3 = "Answer: "

column_names1 = "passage"
column_names2 = "question"
column_names3 = "answer"

def set_header1(header_value):
  global header1
  header1=header_value

def set_header2(header_value):
  global header2
  header2=header_value

def set_header3(header_value):
  global header3
  header3=header_value

def set_column_names1(column_names):
  global column_names1
  column_names1=column_names

def set_column_names2(column_names):
  global column_names2
  column_names2=column_names

def set_column_names3(column_names):
  global column_names3
  column_names3=column_names

def set_master_columns(col1, col2, col3, header_val1, header_val2, header_val3):
  global column_names1, column_names2, column_names3, header1, header2, header3
  column_names1=col1
  column_names2=col2
  column_names3=col3

  header1=header_val1
  header2=header_val2
  header3=header_val3

In [ ]:
# prepping the dataset
raw_datasets_base = load_dataset("flowaicom/HaluEval")#https://huggingface.co/datasets/flowaicom/HaluEval

url = "https://datasets-server.huggingface.co/rows"

all_rows = []
offset = 0
batch_size = 100



while len(all_rows) < 2000:
    params = {
        "dataset": "casehold/casehold",
        "config": "fold_1",
        "split": "train",
        "offset": offset,
        "length": batch_size
    }

    response = requests.get(url, params=params)
    data = response.json()

    batch = [row["row"] for row in data["rows"]]

    if not batch:  # no more data
        break

    all_rows.extend(batch)
    offset += batch_size


# Convert list of rows into Hugging Face Dataset
raw_datasets_legal = Dataset.from_list(all_rows) #https://huggingface.co/datasets/casehold/casehold/viewer/fold_1

def label_map(example):
    if example["label"] == "PASS" or example["label"] == "true" or example["label"] == "1" or example["label"] == 1:
        example["label"] = 1
    else:
        example["label"] = 0
    return example


raw_datasets_base = raw_datasets_base.map(label_map)
raw_datasets_legal= raw_datasets_legal.map(label_map)

# convert legal columns to match HaluEval
def convert_legal(example):
    return {
        "passage": example["citing_prompt"],      # citing_prompt -> passage
        "question": "What is something that is supported by the content?",
        "answer": example["holding_1"],     # holding -> answer
        "label": example["label"]
    }
raw_datasets_legal = raw_datasets_legal.map(convert_legal)

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            header1 + (q if q is not None else ""),
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for q, p, a in zip(examples[column_names1], examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from datasets import Value
from datasets import DatasetDict
cols = ["passage", "question", "answer", "label"]

base_clean = raw_datasets_base.remove_columns(
    [c for c in raw_datasets_base["test"].column_names if c not in cols]
)

legal_clean = raw_datasets_legal.remove_columns(
    [c for c in raw_datasets_legal.column_names if c not in cols]
)

legal_clean = legal_clean.cast_column("label", Value("int64"))
base_clean = base_clean.cast_column("label", Value("int64"))

combined_train = concatenate_datasets([
    base_clean["test"],
    legal_clean
])

combined_dataset = DatasetDict({
    "train": combined_train
})



Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
tokenized_datasets = combined_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

In [ ]:
split_datasets = tokenized_datasets["train"].train_test_split(test_size=0.2, seed=42)

In [ ]:
set_master_columns("question", "passage", "answer", "Question: ", "Passage: ", "Answer: ")


small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = split_datasets["test"].shuffle(seed=42).select(range(100))

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

#added average binary because it tells the model to Treat this as a binary classification problem and focus on the positive class (1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train and evalute the results for base and legal
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.395644,0.900000,0.840909,0.925000,0.880952
2,No log,0.188672,0.950000,0.926829,0.950000,0.938272
3,No log,0.153506,0.960000,0.928571,0.975000,0.951220
4,No log,0.248280,0.950000,0.926829,0.950000,0.938272
5,No log,0.281373,0.940000,0.904762,0.950000,0.926829
6,No log,0.339368,0.910000,0.844444,0.950000,0.894118
7,No log,0.318398,0.940000,0.904762,0.950000,0.926829
8,No log,0.321088,0.940000,0.904762,0.950000,0.926829


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.32108768820762634,
 'eval_accuracy': 0.94,
 'eval_precision': 0.9047619047619048,
 'eval_recall': 0.95,
 'eval_f1': 0.926829268292683,
 'eval_runtime': 0.7846,
 'eval_samples_per_second': 127.448,
 'eval_steps_per_second': 16.568,
 'epoch': 8.0}

In [ ]:

ds = load_dataset("Cleanlab/FinQA-hallucination-detection") #https://huggingface.co/datasets/Cleanlab/FinQA-hallucination-detection

README.md: 0.00B [00:00, ?B/s]

finqa-hallucination-detection.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1657 [00:00<?, ? examples/s]

In [ ]:
#switch to a different data set for evalute
set_master_columns("passage", "question", "answer", "Passage: ", "Question: ", "Answer: ")

raw_eval = ds.rename_columns({
    "query": "question",
    "context": "passage",
    "llm_response": "answer",
    "is_correct": "label"
})
raw_eval=raw_eval.map(tokenize_function, batched=True)

small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = raw_eval["train"].shuffle(seed=42).select(range(400))
from collections import Counter

print(Counter(small_train_dataset["label"]))

Map:   0%|          | 0/1657 [00:00<?, ? examples/s]

Counter({0: 229, 1: 171})


In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train and evalute with a different test set
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.741800,0.542500,0.870370,0.548105,0.672630
2,No log,0.433311,0.845000,0.857506,0.982507,0.915761
3,No log,0.453594,0.852500,0.856784,0.994169,0.920378
4,No log,0.577101,0.857500,0.857500,1.000000,0.923284
5,No log,0.687152,0.820000,0.853786,0.953353,0.900826
6,No log,0.678400,0.852500,0.856784,0.994169,0.920378
7,No log,0.700434,0.852500,0.856784,0.994169,0.920378
8,No log,0.705623,0.852500,0.856784,0.994169,0.920378


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.7056233286857605,
 'eval_accuracy': 0.8525,
 'eval_precision': 0.8567839195979899,
 'eval_recall': 0.9941690962099126,
 'eval_f1': 0.9203778677462888,
 'eval_runtime': 2.9916,
 'eval_samples_per_second': 133.706,
 'eval_steps_per_second': 16.713,
 'epoch': 8.0}

In [ ]:
#We train on base and test on legal so we can see the difference
tokenized_datasets_base = base_clean.map(tokenize_function, batched=True)
tokenized_datasets_legal = legal_clean.map(tokenize_function, batched=True)

set_master_columns("question", "passage", "answer", "Question: ", "Passage: ", "Answer: ")

small_train_dataset = tokenized_datasets_base["test"].shuffle(seed=42).select(range(400))
small_eval_dataset = tokenized_datasets_legal.shuffle(seed=42).select(range(100))

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.616269,0.630000,0.047619,0.055556,0.051282
2,No log,1.067464,0.520000,0.125000,0.277778,0.172414
3,No log,1.983903,0.350000,0.169014,0.666667,0.269663
4,No log,2.125027,0.440000,0.183333,0.611111,0.282051
5,No log,2.384516,0.430000,0.180328,0.611111,0.278481
6,No log,2.058817,0.520000,0.083333,0.166667,0.111111
7,No log,2.411883,0.430000,0.132075,0.388889,0.197183
8,No log,2.459903,0.430000,0.132075,0.388889,0.197183


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 2.4599032402038574,
 'eval_accuracy': 0.43,
 'eval_precision': 0.1320754716981132,
 'eval_recall': 0.3888888888888889,
 'eval_f1': 0.19718309859154928,
 'eval_runtime': 0.7867,
 'eval_samples_per_second': 127.119,
 'eval_steps_per_second': 16.525,
 'epoch': 8.0}

In [ ]:
from datasets import concatenate_datasets
label_0 = tokenized_datasets_legal.filter(lambda x: x["label"] == 0)
label_1 = tokenized_datasets_legal.filter(lambda x: x["label"] == 1)

min_size = min(len(label_0), len(label_1))

label_0 = label_0.shuffle(seed=42).select(range(min_size))
label_1 = label_1.shuffle(seed=42).select(range(min_size))

balanced_set = concatenate_datasets([label_0, label_1]).shuffle(seed=42)

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
#next step legal train and legal evaluate

legal_split_datasets = balanced_set.train_test_split(test_size=0.2, seed=42)

small_train_dataset = legal_split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = legal_split_datasets["test"].shuffle(seed=42).select(range(100))

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.627994,0.690000,0.761905,0.603774,0.673684
2,No log,0.623240,0.680000,0.684211,0.735849,0.709091
3,No log,0.949505,0.590000,0.611111,0.622642,0.616822
4,No log,1.059358,0.710000,0.772727,0.641509,0.701031
5,No log,1.315609,0.700000,0.767442,0.622642,0.687500
6,No log,1.467896,0.680000,0.756098,0.584906,0.659574
7,No log,1.325674,0.690000,0.729167,0.660377,0.693069
8,No log,1.347711,0.690000,0.729167,0.660377,0.693069


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.3477110862731934,
 'eval_accuracy': 0.69,
 'eval_precision': 0.7291666666666666,
 'eval_recall': 0.660377358490566,
 'eval_f1': 0.693069306930693,
 'eval_runtime': 0.7827,
 'eval_samples_per_second': 127.758,
 'eval_steps_per_second': 16.609,
 'epoch': 8.0}

In [ ]:
#no training
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

small_eval_dataset = legal_split_datasets["test"].shuffle(seed=42).select(range(100))
#train
trainer = Trainer(
    model=model, args=training_args, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)

# evaluate WITHOUT training
trainer.evaluate()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': 0.6885982751846313,
 'eval_model_preparation_time': 0.003,
 'eval_accuracy': 0.54,
 'eval_precision': 0.5360824742268041,
 'eval_recall': 0.9811320754716981,
 'eval_f1': 0.6933333333333334,
 'eval_runtime': 0.7906,
 'eval_samples_per_second': 126.481,
 'eval_steps_per_second': 16.443}